<a href="https://colab.research.google.com/github/FabioContrera/criando-agentes-de-ia-com-agno/blob/main/Aula%202/Aula_2_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 2.3 — Tornando o agente mais inteligente com tools

**Curso:** Agno: Criando agentes e sistemas multiagente

**Aula:** 2 — Turbinando agentes com Tools

**Instrutor:** Fabio Contrera

---

## Onde estamos

Na **Aula 2.2** plugamos a primeira tool no Treinador (Tavily, busca web).

Esta aula é sobre **subir o nível**.

1. Adicionar uma **segunda tool** (Wikipedia)
2. Observar o "uso ingênuo" — o agente sem orientação clara pode escolher mal entre as fontes
3. **Refinar instructions** para guiar a escolha entre tools
4. Comparar as versões e validar a melhoria


## Setup

Adicionamos `wikipedia` (lib usada pelo `WikipediaTools` do Agno) à instalação.


In [ ]:
!pip install -U agno openai tavily-python wikipedia

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

---

## Passo 1 — Adicionando uma segunda tool




In [14]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.tavily import TavilyTools
from agno.tools.wikipedia import WikipediaTools

modelo_openai = OpenAIChat(id="gpt-5.4-mini")

treinador = Agent(
    name="Treinador",
    description="Assistente do ScoutAI FC sobre a Seleção Brasileira masculina de futebol.",
    model= modelo_openai,
    instructions=[
        "Você é o Treinador, assistente do ScoutAI FC dedicado à Seleção Brasileira masculina.",
        "Responda em português do Brasil, com tom profissional e analítico.",
        "Quando não tiver certeza de um dado, diga claramente.",
    ],
    tools=[TavilyTools(), WikipediaTools()],   # ← duas tools agora
    markdown=True,
)

---

## Passo 2 — Observando o "uso ingênuo"

Vamos fazer **uma pergunta híbrida** com cara de comissão técnica trabalhando — daquelas que misturam três tipos de necessidade de informação:

> *"Como Telê Santana montava o ataque da Seleção de 82, e quem hoje teria perfil parecido na atual convocação?"*

Por que essa pergunta é interessante:

- **"Telê Santana, Seleção de 82"** → fato histórico consolidado → ideal pra **Wikipedia**
- **"Quem hoje na atual convocação"** → informação que muda → ideal pra **Tavily**
- **"Perfil parecido"** → análise interpretativa → **não precisa de tool nenhuma**

Três necessidades distintas em uma única pergunta. Vamos ver como o agente, **sem orientação**, lida com isso:


In [15]:
treinador.print_response(
    "Como Telê Santana montava o ataque da Seleção de 82, e quem hoje teria perfil parecido na última convocação?",
    stream=True,
)

Output()

INFO Searching wikipedia for: Seleção Brasileira 1982 Telê Santana ataque Falcão Zico Sócrates Júnior Éder Careca  
     escalação tática

ERROR    Error searching Wikipedia for 'Seleção Brasileira 1982 Telê Santana ataque Falcão Zico Sócrates Júnior    
         Éder Careca escalação tática': Page id "Seleção Brasileira 1982 Telê Santana ataque Falcão Zico Sócrates  
         Júnior Éder Careca escalação tática" does not match any pages. Try another id!

INFO Searching wikipedia for: Telê Santana Seleção Brasileira 1982

ERROR    Error searching Wikipedia for 'Telê Santana Seleção Brasileira 1982': Page id "tell santana seleção       
         brasileira 1992" does not match any pages. Try another id!

INFO Searching wikipedia for: Brasil na Copa do Mundo de 1982 escalação

ERROR    Error searching Wikipedia for 'Brasil na Copa do Mundo de 1982 escalação': Page id "brazil na copa do     
         mundo de 1982 escalando" does not match any pages. Try another id!

---

## Passo 3 — Refinando instructions para guiar a escolha entre tools

A correção é uma instruction nova, **explícita sobre quando usar cada fonte**. Não é "use tools", é "use **esta** tool para **isso**, **aquela** para **aquilo**":


In [16]:
treinador = Agent(
    name="Treinador",
    description="Assistente do ScoutAI FC sobre a Seleção Brasileira masculina de futebol.",
    model=modelo_openai,
    instructions=[
        "Você é o Treinador, assistente do ScoutAI FC dedicado à Seleção Brasileira masculina.",
        "Responda em português do Brasil, com tom profissional e analítico.",
        "Quando não tiver certeza de um dado, diga claramente.",

        # Instruction central da aula: política de uso de tools
        "POLÍTICA DE FONTES — siga rigorosamente:",
        "• Para EVENTOS RECENTES (últimos jogos, convocações, lesões, forma atual): "
        "use Tavily (busca web).",
        "• Para FATOS HISTÓRICOS CONSOLIDADOS (Copas antigas, biografias, técnicos do passado, "
        "regulamentos): use Wikipedia — é mais estruturada e citável.",
        "• Para PERGUNTAS CONCEITUAIS (táticas, regras, análise interpretativa): "
        "responda direto sem tool.",
        "• Quando a pergunta tiver MÚLTIPLAS NECESSIDADES, combine fontes: "
        "Wikipedia para a parte histórica, Tavily para a parte recente, "
        "e sua análise para a parte interpretativa.",
    ],
    tools=[TavilyTools(), WikipediaTools()],
    markdown=True,
)

---

## Passo 4 — Validando a melhoria

Antes de testar a versão refinada, vale fazer uma **comparação tripla** com a mesma pergunta híbrida. Três versões do agente, três qualidades de resposta:

1. **Sem tool nenhuma** — o agente responde só com o que aprendeu no treinamento
2. **Com tools, sem política** (o "uso ingênuo" do Passo 2) — temos a resposta acima
3. **Com tools e política refinada** — agora vamos rodar


### Versão 1: agente sem nenhuma tool


In [17]:
# Agente sem tools — só conhecimento interno do modelo
treinador_sem_tool = Agent(
    name="Treinador",
    description="Assistente do ScoutAI FC sobre a Seleção Brasileira masculina de futebol.",
    model=modelo_openai,
    instructions=[
        "Você é o Treinador, assistente do ScoutAI FC dedicado à Seleção Brasileira masculina.",
        "Responda em português do Brasil, com tom profissional e analítico.",
        "Quando não tiver certeza de um dado, diga claramente.",
    ],
    markdown=True,
)

treinador_sem_tool.print_response(
    "Como Telê Santana montava o ataque da Seleção de 82, e quem hoje teria perfil parecido na última convocação?",
    stream=True,
)

Output()

### Versão 2: agente com tools, mas sem política

Mesmo agente do Passo 2 — duas tools (Tavily e Wikipedia), nenhuma orientação sobre **quando usar cada uma**. Vamos rodar a pergunta híbrida outra vez para ter o resultado lado a lado:


In [18]:
# Recriando o agente do Passo 2 — duas tools, sem política
treinador_sem_politica = Agent(
    name="Treinador",
    description="Assistente do ScoutAI FC sobre a Seleção Brasileira masculina de futebol.",
    model=modelo_openai,
    instructions=[
        "Você é o Treinador, assistente do ScoutAI FC dedicado à Seleção Brasileira masculina.",
        "Responda em português do Brasil, com tom profissional e analítico.",
        "Quando não tiver certeza de um dado, diga claramente.",
    ],
    tools=[TavilyTools(), WikipediaTools()],
    markdown=True,
)

treinador_sem_politica.print_response(
    "Como Telê Santana montava o ataque da Seleção de 82, e quem hoje teria perfil parecido na atual convocação?",
    stream=True,
)

Output()

INFO Searching wikipedia for: Seleção Brasileira de Futebol na Copa do Mundo de 1982 Telê Santana ataque jogadores 
     escalacao wikipedia

ERROR    Error searching Wikipedia for 'Seleção Brasileira de Futebol na Copa do Mundo de 1982 Telê Santana ataque 
         jogadores escalacao wikipedia': Page id "Seleção Brasileira de Futebol na Copa do Mundo de 1982 Telê      
         Santana ataque jogadores escalacao wikipedia" does not match any pages. Try another id!

INFO Searching wikipedia for: Seleção Brasileira na Copa do Mundo de 1982

INFO Searching wikipedia for: Telê Santana 1982 Seleção Brasileira

ERROR    Error searching Wikipedia for 'Telê Santana 1982 Seleção Brasileira': Page id "tell santana 1992 seleção  
         brasileira" does not match any pages. Try another id!

INFO Searching wikipedia for: Brasil 1982 Telê Santana formação 4-2-2-2

ERROR    Error searching Wikipedia for 'Brasil 1982 Telê Santana formação 4-2-2-2': Page id "brazil 1992 telê      
         santana formação 4 2 2 2" does not match any pages. Try another id!

### Versão 3: agente com tools e política refinada

Mesma pergunta híbrida, agora com a política de fontes ativada. Observe nas chamadas de tool: o agente deve acionar **as duas** (Wikipedia para Telê e a Seleção de 82, Tavily para a convocação atual) e usar conhecimento próprio para a parte interpretativa.


In [19]:
treinador.print_response(
    "Como Telê Santana montava o ataque da Seleção de 82, e quem hoje teria perfil parecido na última convocação?",
    stream=True,
)

Output()

INFO Searching wikipedia for: Seleção Brasileira de Futebol na Copa do Mundo FIFA de 1982 Telê Santana ataque      
     titulares Zico Sócrates Falcão Éder Serginho TV

ERROR    Error searching Wikipedia for 'Seleção Brasileira de Futebol na Copa do Mundo FIFA de 1982 Telê Santana   
         ataque titulares Zico Sócrates Falcão Éder Serginho TV': Page id "Seleção Brasileira de Futebol na Copa do
         Mundo FIFA de 1982 Telê Santana ataque titulares Zico Sócrates Falcão Éder Serginho TV" does not match any
         pages. Try another id!

INFO Searching wikipedia for: Seleção Brasileira de Futebol na Copa do Mundo FIFA de 1982

INFO Searching wikipedia for: Telê Santana

INFO Searching wikipedia for: 1982 FIFA World Cup Brazil squad formation 4-2-2-2

### Comparando as três versões

| Versão | O que provavelmente aconteceu |
|---|---|
| **Sem tool** | Resposta inteira "de cabeça" — bem escrita, mas com risco de inventar nomes da convocação atual e datar mal os fatos da Seleção de 82 |
| **Com tools, sem política** | Acionou alguma tool, mas talvez a errada (Tavily pra histórico de 82) ou só uma quando precisava de duas |
| **Com tools e política** | Wikipedia para Telê e a Seleção de 82, Tavily para a convocação atual, análise interpretativa direto |



### E para confirmar que ele também sabe *não* usar tool

Pergunta puramente tática — não deve aparecer chamada de tool nenhuma:


In [20]:
treinador.print_response(
    "Em que cenários um esquema com 3 zagueiros é mais eficaz que um com 4?",
    stream=True,
)

Output()

---

## O custo de ter múltiplas tools

Adicionar tools resolve qualidade — mas não é grátis. Vale revisitar os trade-offs da Aula 2.1 agora no contexto específico de **múltiplas tools**:

- **Latência multiplica.**

- **Custo por turno aumenta.** .

- **Probabilidade de falha cresce.**

### Decisão prática

Antes de adicionar tool nova ao agente, pergunte:

- A **qualidade** que ela traz justifica a **latência** que ela adiciona?
- Existe **categoria de pergunta** que se beneficia dela, ou estou adicionando "por garantia"?
- Se a tool falhar, o agente **degrada com graça** (responde com o que tem) ou trava?



---

## Como identificar falhas no uso de tools

Quatro sintomas a observar nos logs do agente:

| Sintoma | Como identificar | Como corrigir |
|---|---|---|
| **Tool não foi acionada quando deveria** | Resposta com confiança sobre dado verificável, mas sem `Running: tool(...)` no log | Refinar instruction com exemplo do tipo de pergunta que exige tool |
| **Tool foi acionada quando não precisava** | Latência alta em pergunta conceitual, com `Running: tool(...)` desnecessário | Refinar instruction listando explicitamente o que NÃO precisa de tool |
| **Tool errada foi escolhida** | Wikipedia chamada para evento de ontem, ou Tavily para fato de 1958 | Reescrever política conectando **categoria de pergunta** a **tool específica** |
| **Tool falhou e agente improvisou** | `ERROR No results found` no log + resposta confiante mesmo assim | Adicionar instruction defensiva: "se a tool falhar, admita explicitamente em vez de chutar" |

### O loop de correção

Em qualquer agente em produção, o ciclo de melhoria de tools é o mesmo:

```
1. Coletar amostras de execuções reais (logs)
2. Identificar qual dos 4 sintomas aparece
3. Reescrever a instruction da política
4. Rodar testes comparativos (versão antiga vs nova)
5. Repetir
```


---

## Recapitulação da Aula 2

A jornada da Aula 2 inteira, em três frases:

- **2.1** — Vimos por que agentes precisam de tools (alucinação confiante e desatualização)
- **2.2** — Plugamos a primeira tool e vimos o agente decidindo entre usá-la ou não
- **2.3** — Adicionamos múltiplas tools e ensinamos o agente a **escolher entre fontes** com uma política explícita

### Estado atual do Treinador

```
Treinador (estado atual)
│
├── ✅ Identidade e tom (Aula 1.3)
├── ✅ Interface de chat (Aula 1.3)
├── ✅ Acesso a busca web (Aula 2.2)
├── ✅ Múltiplas fontes com política explícita (Aula 2.3 ← você está aqui)
│
├── ❌ Faz tudo sozinho                  → Aula 3 (Olheiro + Analista)
├── ❌ Sem fluxo determinístico          → Aula 4 (Workflows)
└── ❌ Sem produção / monitoramento      → Aula 5 (Agent OS)
```


---

## Aplicando o que aprendemos em outros contextos


| Domínio | Tools típicas | Pergunta típica que precisa de política de fontes |
|---|---|---|
| **Análise financeira** | API de cotações, base de relatórios, calculadora, gerador de gráficos | "Compare o desempenho dessas 3 ações em 2024." (cotações + gráfico) |
| **Saúde (assistente para profissional)** | Base de protocolos clínicos (RAG), bula online, busca em literatura científica | "Qual a interação entre esses dois remédios e há estudos recentes?" (bula + PubMed) |
| **Educação** | Banco de questões interno, busca web, calculadora simbólica, gerador de gráficos | "Resolve essa equação e mostra o gráfico." (calculadora + gráfico) |
| **Logística** | API de rastreamento, base de rotas, calculadora de distâncias | "Otimize essa entrega considerando trânsito atual." (rotas + tempo real) |
| **Jurídico** | Base de jurisprudência (RAG), busca em legislação, gerador de minutas | "Há precedentes sobre esse tipo de caso?" (RAG + busca web) |



---

## Próxima aula

**Aula 3 — Times de IA: como fazer agentes trabalharem juntos**

Note como o Treinador, no fim desta aula, está fazendo **muita coisa**: busca informação, escolhe fonte, analisa, sintetiza.

A próxima aula resolve isso introduzindo **especialistas**: o **Olheiro** (que vai herdar as tools de busca) e o **Analista** (que interpreta dados — e ganha a primeira **tool customizada do curso**, escrita do zero pra esse papel).
